In [1]:
"""

Prepare the masks of the ice shelves based on BedMachine

"""

'\n\nPrepare the masks of the ice shelves based on BedMachine\n\n'

In [2]:
import xarray as xr
import numpy as np
from pyproj import Transformer
import pandas as pd
from tqdm.notebook import trange, tqdm
import cc3d
import matplotlib as mpl


import multimelt.plume_functions as pf
import multimelt.box_functions as bf
import multimelt.useful_functions as uf
import multimelt.create_isf_mask_functions as isfmf

import distributed

In [3]:
%matplotlib inline

In [4]:
map_lim = [-3000000,3000000]

#chunk_size = 700
chunk_size = False

In [5]:
nemo_run = 'OPM018'

READ IN DATA

In [6]:
######
###### READ IN DATA
######


#if run on luke
inputpath_IMBIE='../../../../../../burgardc/DATA/'
inputpath_data='../../../../../../burgardc/SCRIPTS/basal_melt_param/data/interim/NEMO_eORCA025.L121_'+nemo_run+'_ANT_STEREO/'
inputpath_metadata='../../../../../../burgardc/SCRIPTS/basal_melt_param/data/raw/MASK_METADATA/'
#outputpath_mask='/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'
outputpath_mask='../../../data/interim/ANTARCTICA_IS_MASKS/nemo_5km_'+nemo_run+'/'
#outputpath_mask_orig='/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/ANTARCTICA_IS_MASKS/nemo_5km/'
#outputpath_boxes = '/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/BOXES/nemo_5km_'+nemo_run+'/'
#outputpath_plumes = '/bettik/burgardc/SCRIPTS/basal_melt_param/data/interim/PLUMES/nemo_5km_'+nemo_run+'/'

file_mask_orig = xr.open_dataset(inputpath_data+'other_mask_vars_Ant_stereo.nc')
file_mask_orig_cut = uf.cut_domain_stereo(file_mask_orig, map_lim, map_lim)
file_mask = xr.open_dataset(inputpath_data+'custom_lsmask_Ant_stereo_clean.nc')#, chunks={'x': chunk_size, 'y': chunk_size})
file_mask_cut = uf.cut_domain_stereo(file_mask, map_lim, map_lim)
file_other = xr.open_dataset(inputpath_data+'corrected_draft_bathy_isf.nc')#, chunks={'x': chunk_size, 'y': chunk_size})
file_other_cut = uf.cut_domain_stereo(file_other, map_lim, map_lim)
file_conc = xr.open_dataset(inputpath_data+'isfdraft_conc_Ant_stereo.nc')
file_conc_cut = uf.cut_domain_stereo(file_conc, map_lim, map_lim)
#ds_nemo = xr.open_dataset(outputpath_mask_orig+'nemo_5km_isf_masks_and_info_and_distance.nc')

In [7]:
file_bed_orig = file_mask_orig_cut['bathy_metry']
file_draft = file_other_cut['corrected_isfdraft'] 
file_msk = file_mask_cut['ls_mask012'] #0 = ocean, 1 = ice shelves, 2 = grounded ice
file_isf_conc = file_conc_cut['isfdraft_conc']

xx = file_mask_cut['x']
yy = file_mask_cut['y']


In [8]:
mask_IMBIE = xr.open_dataset(inputpath_IMBIE + 'Mask_Iceshelf_IMBIE2_v2_5km.nc')
mask_IMBIE['Iceshelf_extrap'] = mask_IMBIE['Iceshelf_extrap'].where(mask_IMBIE['Iceshelf_extrap'] != 1, 134) # because 1 will be ocean
mask_IMBIE_containing_names = xr.open_dataset(inputpath_IMBIE + 'Mask_Iceshelf_IMBIE2_v2.nc')
mask_IMBIE_containing_names_corrected = xr.concat([mask_IMBIE_containing_names['NAME'].drop_sel(ID=1),xr.DataArray(data=['Jelbart'], dims=['ID']).assign_coords({'ID': [134]})], dim='ID') # because 1 will be ocean

In [9]:

whole_ds = isfmf.create_mask_and_metadata_isf(file_msk, -1*file_bed_orig, file_msk, -1*file_draft, file_isf_conc, False, 
                                          inputpath_IMBIE + 'Mask_Iceshelf_IMBIE2_v2_5km.nc', outputpath_mask, 
                                          mask_IMBIE_containing_names_corrected, 
                                          inputpath_metadata+'GL_flux_rignot13.csv', mouginot_basins=True, variable_geometry=False,
                                          write_ismask = 'yes', write_groundmask = 'yes', write_outfile='yes',
                                          ground_point ='no',dist=40, add_fac=250, connectivity=4, threshold=4,
                                          write_metadata='yes', AlexIslandisf = [101,102,103,105,106,107,109])

# Write to netcdf
print('------- WRITE TO NETCDF -----------')
whole_ds.to_netcdf(outputpath_mask + 'BedMachinev2_4km_isf_masks_and_info_and_distance_oneFRIS.nc','w')


--------- PREPARE THE MASKS --------------
handling coordinates
Reading in latlon boundaries
Define the regional masks
Define ground_mask
Distance to South Pole for initial ground square = 40
Additional number of iterations = 250


  0%|          | 0/859 [00:00<?, ?it/s]

Define grounding line
Grounding line is the first point on the ground: no
Define front
Define pinning points


  0%|          | 0/25 [00:00<?, ?it/s]

Define pinning point boundaries
Merge into one netcdf
--------- PREPARE THE METADATA --------------
Prepare csv with metadata
--------- COMBINE MASK AND METADATA --------------
--------- COMPUTE DISTANCE TO GROUNDING LINE AND ICE FRONT --------------


/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:1032: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['front_max_lon'].loc[10] = lon.where(mask_front == 10).where(lon < -100).max()
/bettik/ockendeh/SCRIPTS/multimelt/multimelt/create_isf_mask_functions.py:1033: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['front_min_lon'].loc[10] = lon.where(mask_front == 10).where(lon > 100).min()


  0%|          | 0/118 [00:00<?, ?it/s]

------- WRITE TO NETCDF -----------


In [10]:
whole_ds   

<xarray.Dataset>
Dimensions:              (y: 1200, x: 1200, Nisf: 118)
Coordinates:
    longitude            (y, x) float64 -45.0 -44.95 -44.9 ... 135.1 135.0 135.0
    latitude             (y, x) float64 -52.34 -52.37 -52.4 ... -52.38 -52.35
  * x                    (x) float64 -2.998e+06 -2.993e+06 ... 2.997e+06
  * y                    (y) float64 2.998e+06 2.993e+06 ... -2.997e+06
  * Nisf                 (Nisf) int64 2 3 4 5 6 7 8 ... 129 130 131 132 133 134
Data variables: (12/18)
    ISF_mask             (y, x) float32 1.0 1.0 1.0 1.0 1.0 ... 1.0 1.0 1.0 1.0
    GL_mask              (y, x) float64 nan nan nan nan nan ... nan nan nan nan
    IF_mask              (y, x) float64 nan nan nan nan nan ... nan nan nan nan
    PP_mask              (y, x) float32 nan nan nan nan nan ... nan nan nan nan
    ground_mask          (y, x) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    isf_name             (Nisf) object 'Fimbul' '' 'Vigrid' ... 'Atka' 'Jelbart'
    ...                   ...
    front_max_lat        (Nisf) float64 -69.61 nan -69.94 ... -70.42 -70.34
    front_min_lon        (Nisf) float64 -2.72 nan 7.783 ... -9.979 -7.664 -5.968
    front_max_lon        (Nisf) float64 7.445 nan 8.964 ... -7.816 -6.367 -3.551
    dGL                  (y, x) float64 nan nan nan nan nan ... nan nan nan nan
    dIF                  (y, x) float64 nan nan nan nan nan ... nan nan nan nan
    dGL_dIF              (y, x) float64 nan nan nan nan nan ... nan nan nan nan
Attributes:
    history:     Created with combine_mask_metadata() by C. Burgard. dGL, dIF...
    projection:  Polar Stereographic South (71S,0E)
    proj4:       +init=epsg:3031
    Note:        isf ID and individual isf characteristics can be found in ic...

In [11]:
whole_ds[whole_ds['isf_name']== 'Helen']

ValueError: Unsupported key-type <class 'xarray.core.dataarray.DataArray'>

In [ ]:
whole_ds.